# 03 交叉熵损失 F.cross_entropy

以“是否有行人”二分类为例：类别 `0 = 无行人`，类别 `1 = 有行人`。

要点：
- `F.cross_entropy` 接收**原始分数 logits**（形状 `(N, C)`，C 为类别数），**不要**先 softmax。
- 标签是**整数类别编号**（长整型），形状 `(N,)`。
- 一个样本的概率向量是 `(2,)`，两个类别。
- 本题演示：如何把指定概率转换成等价 logits（`log(p)`），以及概率为 0 时损失为 `inf`。

In [1]:
import torch
import torch.nn.functional as F

### 情况1：预测正确（置信 0.9）
模型给出概率 `[0.1, 0.9]`（无=0.1，有=0.9），真实类别是 `1`（有行人）。

In [2]:
# 把概率转成等价 logits：logits = log(p)
logits = torch.log(torch.tensor([[0.1, 0.9]]))  # shape (1, 2)
label = torch.tensor([1])                       # 真实类别：有行人
loss = F.cross_entropy(logits, label)
print('logits =', logits)
print('正确预测的损失 =', loss.item(), ' （约等于 -log(0.9) = 0.105）')

logits = tensor([[-2.3026, -0.1054]])
正确预测的损失 = 0.10536056011915207  （约等于 -log(0.9) = 0.105）


### 情况2：预测错误（同样的预测，真实类别是 0）
模型仍给 `[0.1, 0.9]`，但真实是 `0`（无行人）。

In [3]:
logits = torch.log(torch.tensor([[0.1, 0.9]]))
label = torch.tensor([0])   # 真实类别：无行人（预测反了）
loss = F.cross_entropy(logits, label)
print('错误预测的损失 =', loss.item(), ' （约等于 -log(0.1) = 2.303）')

错误预测的损失 = 2.3025851249694824  （约等于 -log(0.1) = 2.303）


### 情况3：真实类别的概率为 0
模型把类别 0 的概率直接给成 `0.0`，而真实类别正是 `0`。

In [4]:
# log(0) = -inf，这是预期行为（会有一条 warning）
logits = torch.log(torch.tensor([[0.0, 1.0]]))
label = torch.tensor([0])
loss = F.cross_entropy(logits, label)
print('概率为 0 时的损失 =', loss.item(), ' （inf = -log(0)，预期结果）')

概率为 0 时的损失 = inf  （inf = -log(0)，预期结果）


### 情况4：概率为 1 且预测正确
模型给 `[0.0, 1.0]`，真实类别是 `1`。

In [5]:
logits = torch.log(torch.tensor([[0.0, 1.0]]))
label = torch.tensor([1])
loss = F.cross_entropy(logits, label)
print('概率为 1 且正确时的损失 =', loss.item(), ' （-log(1) = 0）')

概率为 1 且正确时的损失 = 0.0  （-log(1) = 0）


## 观察

1. **正确 vs 错误**：正确预测损失小（`0.105`），错误预测损失大（`2.303`）——交叉熵对错误预测惩罚更重。
2. **传 logits 不传概率**：`F.cross_entropy` 内部会做 `log_softmax`，所以传入的是原始分数；如果先 softmax 再传反而会算错。
3. **概率为 0 → inf**：当真实类别的预测概率为 0 时，`-log(0) = inf`，这就是训练中要避免出现 `log(0)` 的原因（实际训练会用 logits 加极小值或 log-softmax 技巧）。
4. **概率为 1 且正确 → 0**：模型 100% 确信且猜对时，损失为 0。
5. **概率转 logits**：用 `torch.log(p)` 即可把概率向量还原成等价的原始分数。